# P3 — Validación de Índices Compuestos Tácticos

**Jugador objetivo:** Novak Djokovic  
**Jugador rival:** Taylor Fritz (diestro, sacador agresivo, Top 10 Hard court)  
**Fuente:** JeffSackmann/tennis_MatchChartingProject (CC BY-NC-SA 4.0)  
**Objetivo:** Calcular y validar los 4 índices compuestos del P3 antes de implementarlos en DAX.

**Índices calculados:**
- IA (Índice de Amenaza): qué daño hace Djokovic con cada patrón
- IV (Índice de Vulnerabilidad): dónde Fritz puede explotar a Djokovic
- PA (Prioridad de Acción): qué debe hacer Fritz primero
- NR (Nivel de Riesgo): coste de error en cada acción

**Referencia de medidas P2 reutilizadas (nombres exactos en Power BI):**
- `% Pts Ganados 1er Saque` → 73.26%
- `% Pts Ganados 2do Saque` → 54.93%
- `% Pts Ganados Al Saque` → 66.89%
- `% Pts Ganados Al Resto` → 40.47%
- `% BP Salvados` → 63.99% (desde overview)
- `% BP Convertidos` → 42.74%
- `% Pts Ganados En Red` → 70.21%
- `% Pts Rally Corto` → 33.90%
- `% Restos Profundos` → 84.17%
- `% Pts Ganados Resto En Juego` → 52.57%

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

# Cargar todas las tablas necesarias
matches      = pd.read_csv(f"{DATA_DIR}/charting-m-matches.csv", encoding="utf-8")
overview     = pd.read_csv(f"{DATA_DIR}/charting-m-stats-Overview.csv", encoding="utf-8")
serve_basics = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ServeBasics.csv", encoding="utf-8")
return_depth = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ReturnDepth.csv", encoding="utf-8")
return_out   = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ReturnOutcomes.csv", encoding="utf-8")
net_points   = pd.read_csv(f"{DATA_DIR}/charting-m-stats-NetPoints.csv", encoding="utf-8")
kp_serve     = pd.read_csv(f"{DATA_DIR}/charting-m-stats-KeyPointsServe.csv", encoding="utf-8")
kp_return    = pd.read_csv(f"{DATA_DIR}/charting-m-stats-KeyPointsReturn.csv", encoding="utf-8")
resultados   = pd.read_csv(f"{DATA_DIR}/djokovic_resultados.csv", encoding="utf-8")

# Limpiar fila corrupta de matches
ID_CORRUPTO = "20240915-M-Davis_Cup_World_Group-RR-Botic_Van_De_Zandschulp-Matteo_Berrettini"
matches = matches[matches["match_id"] != ID_CORRUPTO].copy()

# Superficies válidas
VALID_SURF = ["Clay", "Grass", "Hard"]
matches = matches[matches["Surface"].isin(VALID_SURF)]

# Filtros Djokovic por tabla
dj_ov    = overview[(overview["player"] == "Novak Djokovic") & (overview["set"] == "Total")]
dj_sb    = serve_basics[serve_basics["player"] == "Novak Djokovic"]
dj_rd    = return_depth[return_depth["player"] == "Novak Djokovic"]
dj_ro    = return_out[return_out["player"] == "Novak Djokovic"]
dj_net   = net_points[(net_points["player"] == "Novak Djokovic") & (net_points["row"] == "NetPoints")]
dj_kps   = kp_serve[kp_serve["player"] == "Novak Djokovic"]
dj_kpr   = kp_return[kp_return["player"] == "Novak Djokovic"]

print("Tablas cargadas correctamente.")
print(f"Partidos únicos en overview (Djokovic, Total): {dj_ov['match_id'].nunique()}")

Tablas cargadas correctamente.
Partidos únicos en overview (Djokovic, Total): 550


## SECCIÓN 1 — Medidas base reutilizadas del P2
Verificación de que los valores coinciden con los del archivo medidas_DAX_P2.txt

In [2]:
# ── Medidas reutilizadas del P2 ─────────────────────────────────────────────

# % Pts Ganados 1er Saque (P2: 73.26%)
pct_1s = dj_ov["first_won"].sum() / dj_ov["first_in"].sum()

# % Pts Ganados 2do Saque (P2: 54.93%)
pct_2s = dj_ov["second_won"].sum() / dj_ov["second_in"].sum()

# % Pts Ganados Al Saque (P2: 66.89%)
pct_saque = (dj_ov["first_won"].sum() + dj_ov["second_won"].sum()) / dj_ov["serve_pts"].sum()

# % Pts Ganados Al Resto (P2: 40.47%)
pct_resto = dj_ov["return_pts_won"].sum() / dj_ov["return_pts"].sum()

# % BP Salvados (P2: 63.99%)
pct_bp_salv = dj_ov["bp_saved"].sum() / dj_ov["bk_pts"].sum()

# % BP Convertidos (P2: 42.74%)
pct_bp_conv = dj_kpr[dj_kpr["row"]=="BPO"]["pts_won"].sum() / dj_kpr[dj_kpr["row"]=="BPO"]["pts"].sum()

# % Pts Ganados En Red (P2: 70.21%)
pct_red = dj_net["pts_won"].sum() / dj_net["net_pts"].sum()

# % Pts Rally Corto (P2: 33.90%)
sb_total = dj_sb[dj_sb["row"] == "Total"]
pct_rally_corto = sb_total["pts_won_lte_3_shots"].sum() / sb_total["pts"].sum()

# % Restos Profundos (P2: 84.17%)
rd_total = dj_rd[dj_rd["row"] == "Total"]
deep_total = rd_total["deep"].sum() + rd_total["very_deep"].sum()
all_depth  = rd_total["shallow"].sum() + rd_total["deep"].sum() + rd_total["very_deep"].sum()
pct_profundo = deep_total / all_depth

# % Pts Ganados Resto En Juego (P2: 52.57%)
ro_total = dj_ro[dj_ro["row"] == "Total"]
pct_inplay = ro_total["in_play_won"].sum() / ro_total["in_play"].sum()

print("=== VERIFICACIÓN MEDIDAS P2 (deben coincidir con medidas_DAX_P2.txt) ===")
verificacion = [
    ("% Pts Ganados 1er Saque",      pct_1s,         0.7326),
    ("% Pts Ganados 2do Saque",      pct_2s,         0.5493),
    ("% Pts Ganados Al Saque",       pct_saque,      0.6689),
    ("% Pts Ganados Al Resto",       pct_resto,      0.4047),
    ("% BP Salvados",                pct_bp_salv,    0.6399),
    ("% BP Convertidos",             pct_bp_conv,    0.4274),
    ("% Pts Ganados En Red",         pct_red,        0.7021),
    ("% Pts Rally Corto",            pct_rally_corto,0.3390),
    ("% Restos Profundos",           pct_profundo,   0.8417),
    ("% Pts Ganados Resto En Juego", pct_inplay,     0.5257),
]
for nombre, calculado, referencia in verificacion:
    delta = abs(calculado - referencia)
    estado = "✓" if delta < 0.0005 else "⚠ DIFERENCIA"
    print(f"  {estado}  {nombre:<40s}: {calculado:.4f}  (ref P2: {referencia:.4f})")

=== VERIFICACIÓN MEDIDAS P2 (deben coincidir con medidas_DAX_P2.txt) ===
  ✓  % Pts Ganados 1er Saque                 : 0.7326  (ref P2: 0.7326)
  ✓  % Pts Ganados 2do Saque                 : 0.5493  (ref P2: 0.5493)
  ✓  % Pts Ganados Al Saque                  : 0.6689  (ref P2: 0.6689)
  ✓  % Pts Ganados Al Resto                  : 0.4047  (ref P2: 0.4047)
  ✓  % BP Salvados                           : 0.6399  (ref P2: 0.6399)
  ✓  % BP Convertidos                        : 0.4274  (ref P2: 0.4274)
  ✓  % Pts Ganados En Red                    : 0.7021  (ref P2: 0.7021)
  ✓  % Pts Rally Corto                       : 0.3390  (ref P2: 0.3390)
  ✓  % Restos Profundos                      : 0.8417  (ref P2: 0.8417)
  ✓  % Pts Ganados Resto En Juego            : 0.5257  (ref P2: 0.5257)


## SECCIÓN 2 — Medidas auxiliares nuevas del P3

In [3]:
# ── % Saque 2S Body (nueva medida P3) ───────────────────────────────────────
# Referencia DAX: % Saque 2S Body
sb_2s = dj_sb[dj_sb["row"] == "2"]
total_2s_dir = sb_2s["wide"].sum() + sb_2s["body"].sum() + sb_2s["t"].sum()
pct_body_2s = sb_2s["body"].sum() / total_2s_dir

# ── % Pts Rally Largo Resto (nueva medida P3) ───────────────────────────────
# row=9 en ReturnOutcomes = puntos que superaron 9 golpes
ro_largo = dj_ro[dj_ro["row"] == "9"]
pct_largo = ro_largo["pts_won"].sum() / ro_largo["pts"].sum()

# ── % Pts Rally Medio Resto (nueva medida P3) ───────────────────────────────
# row=4 y row=6 son los buckets con peor rendimiento de Djokovic al resto
ro_medio = dj_ro[dj_ro["row"].isin(["4", "6"])]
pct_medio = ro_medio["pts_won"].sum() / ro_medio["pts"].sum()

# Desglose por bucket para documentar
ro_4 = dj_ro[dj_ro["row"] == "4"]
ro_6 = dj_ro[dj_ro["row"] == "6"]
pct_4 = ro_4["pts_won"].sum() / ro_4["pts"].sum()
pct_6 = ro_6["pts_won"].sum() / ro_6["pts"].sum()

# ── Caída relativa 1S → 2S ───────────────────────────────────────────────────
caida_rel_1s_2s = (pct_1s - pct_2s) / pct_1s

# ── Proporción de puntos al resto en 9+ golpes (para normalizar IA) ─────────
pts_total_resto = dj_ro[dj_ro["row"] == "Total"]["pts"].sum()
pts_9mas = ro_largo["pts"].sum()
freq_rally_largo = pts_9mas / pts_total_resto

# ── BPO como proporción de puntos clave al resto ────────────────────────────
pts_rtotal = dj_kpr[dj_kpr["row"] == "RTotal"]["pts"].sum()
pts_bpo    = dj_kpr[dj_kpr["row"] == "BPO"]["pts"].sum()
freq_bpo   = pts_bpo / pts_rtotal

print("=== MEDIDAS AUXILIARES NUEVAS P3 ===")
print(f"  % Saque 2S Body                    : {pct_body_2s:.4f}  (espera: 0.3272)")
print(f"  % Pts Rally Largo (row=9)           : {pct_largo:.4f}  (espera: 0.5889)")
print(f"  % Pts Rally Medio (row=4+6 agregado): {pct_medio:.4f}")
print(f"    └─ row=4 solo                     : {pct_4:.4f}  (espera: 0.3594)")
print(f"    └─ row=6 solo                     : {pct_6:.4f}  (espera: 0.3460)")
print(f"  Caída relativa 1S→2S               : {caida_rel_1s_2s:.4f}  (espera: ~0.251)")
print(f"  Freq. puntos 9+ golpes / Total resto: {freq_rally_largo:.4f}")
print(f"  Freq. BPO / RTotal                 : {freq_bpo:.4f}")

=== MEDIDAS AUXILIARES NUEVAS P3 ===
  % Saque 2S Body                    : 0.3272  (espera: 0.3272)
  % Pts Rally Largo (row=9)           : 0.5889  (espera: 0.5889)
  % Pts Rally Medio (row=4+6 agregado): 0.3532
    └─ row=4 solo                     : 0.3594  (espera: 0.3594)
    └─ row=6 solo                     : 0.3460  (espera: 0.3460)
  Caída relativa 1S→2S               : 0.2503  (espera: ~0.251)
  Freq. puntos 9+ golpes / Total resto: 0.2003
  Freq. BPO / RTotal                 : 0.2437


## SECCIÓN 3 — Índice de Amenaza (IA)
Fórmula: `IA = frecuencia_norm × rendimiento_Djokovic × peso_presión`

In [4]:
# ── IA 1: Saque 2S Predecible ────────────────────────────────────────────────
# Frecuencia: % Body en 2S (normalizado sobre máx 0.50)
# Rendimiento: % pts ganados 2S
# Peso presión: 1.0 (situación normal)
ia_2s_freq_norm  = min(pct_body_2s / 0.50, 1.0)
ia_2s_rend       = pct_2s
ia_2s_peso       = 1.0
IA_saque_2s      = round(ia_2s_freq_norm * ia_2s_rend * ia_2s_peso, 3)

# ── IA 2: Rally Largo ────────────────────────────────────────────────────────
# Frecuencia: % puntos que llegan a 9+ golpes (sobre Total)
# Rendimiento: % pts ganados en esos rallies
# Peso presión: 1.2 (Djokovic ya tiene control a 9+ golpes)
ia_largo_freq_norm = min(freq_rally_largo / 0.30, 1.0)
ia_largo_rend      = pct_largo
ia_largo_peso      = 1.2
IA_rally_largo     = round(min(ia_largo_freq_norm * ia_largo_rend * ia_largo_peso, 1.0), 3)

# ── IA 3: Conversión Break Points ────────────────────────────────────────────
# Frecuencia: BPO / RTotal (normalizado sobre máx 0.35)
# Rendimiento: % BP convertidos
# Peso presión: 1.5 (máximo — puntos más decisivos)
ia_bp_freq_norm = min(freq_bpo / 0.35, 1.0)
ia_bp_rend      = pct_bp_conv
ia_bp_peso      = 1.5
IA_bp           = round(min(ia_bp_freq_norm * ia_bp_rend * ia_bp_peso, 1.0), 3)

# ── IA 4: Resto Profundo ─────────────────────────────────────────────────────
# Frecuencia: % restos profundos (normalizado sobre máx 0.90)
# Rendimiento: % pts ganados cuando resto entra en juego
# Peso presión: 1.1 (golpe inaugural — condiciona todo el rally)
ia_prof_freq_norm = min(pct_profundo / 0.90, 1.0)
ia_prof_rend      = pct_inplay
ia_prof_peso      = 1.1
IA_resto_prof     = round(min(ia_prof_freq_norm * ia_prof_rend * ia_prof_peso, 1.0), 3)

print("=== ÍNDICE DE AMENAZA (IA) — cuanto más alto, mayor amenaza para Fritz ===")
print(f"  IA Conversión BP          : {IA_bp:.3f}  → CRÍTICA")
print(f"  IA Resto Profundo         : {IA_resto_prof:.3f}  → MUY ALTA")
print(f"  IA Rally Largo            : {IA_rally_largo:.3f}  → ALTA")
print(f"  IA Saque 2S Predecible    : {IA_saque_2s:.3f}  → MEDIA")
print()
print("Componentes detallados:")
print(f"  IA BP:      freq_norm={ia_bp_freq_norm:.3f} × rend={ia_bp_rend:.4f} × peso={ia_bp_peso} = {IA_bp:.3f}")
print(f"  IA Prof:    freq_norm={ia_prof_freq_norm:.3f} × rend={ia_prof_rend:.4f} × peso={ia_prof_peso} = {IA_resto_prof:.3f}")
print(f"  IA Largo:   freq_norm={ia_largo_freq_norm:.3f} × rend={ia_largo_rend:.4f} × peso={ia_largo_peso} = {IA_rally_largo:.3f}")
print(f"  IA 2S:      freq_norm={ia_2s_freq_norm:.3f} × rend={ia_2s_rend:.4f} × peso={ia_2s_peso} = {IA_saque_2s:.3f}")

=== ÍNDICE DE AMENAZA (IA) — cuanto más alto, mayor amenaza para Fritz ===
  IA Conversión BP          : 0.446  → CRÍTICA
  IA Resto Profundo         : 0.541  → MUY ALTA
  IA Rally Largo            : 0.472  → ALTA
  IA Saque 2S Predecible    : 0.359  → MEDIA

Componentes detallados:
  IA BP:      freq_norm=0.696 × rend=0.4274 × peso=1.5 = 0.446
  IA Prof:    freq_norm=0.935 × rend=0.5257 × peso=1.1 = 0.541
  IA Largo:   freq_norm=0.668 × rend=0.5889 × peso=1.2 = 0.472
  IA 2S:      freq_norm=0.654 × rend=0.5493 × peso=1.0 = 0.359


## SECCIÓN 4 — Índice de Vulnerabilidad (IV)
Fórmula: `IV = frecuencia_suficiente × caída_relativa × ajuste_superficie`

Calculado por superficie. Superficie de referencia para Fritz: Hard.

In [5]:
# Ajuste por superficie
ajuste_sup = {"Hard": 1.0, "Clay": 0.9, "Grass": 0.6}

resultados_iv = {}

for surf, ajuste in ajuste_sup.items():
    # Partidos de esa superficie
    match_ids_surf = matches[matches["Surface"] == surf]["match_id"]
    n_partidos = len(match_ids_surf)

    # Factor confianza muestra
    if n_partidos >= 300:   factor_conf = 1.0
    elif n_partidos >= 100: factor_conf = 0.9
    elif n_partidos >= 30:  factor_conf = 0.7
    elif n_partidos >= 10:  factor_conf = 0.4
    else:                   factor_conf = 0.3

    # Medidas base filtradas por superficie
    ov_s = dj_ov[dj_ov["match_id"].isin(match_ids_surf)]
    sb_s = dj_sb[dj_sb["match_id"].isin(match_ids_surf)]
    kps_s = dj_kps[dj_kps["match_id"].isin(match_ids_surf)]

    # % Pts Al Saque en esa superficie
    pct_saque_s = (ov_s["first_won"].sum() + ov_s["second_won"].sum()) / ov_s["serve_pts"].sum()

    # % Pts Ganados 1S y 2S en esa superficie
    pct_1s_s = ov_s["first_won"].sum() / ov_s["first_in"].sum()
    pct_2s_s = ov_s["second_won"].sum() / ov_s["second_in"].sum()

    # % Pts Rally Corto en esa superficie
    sb_s_tot = sb_s[sb_s["row"] == "Total"]
    pct_corto_s = sb_s_tot["pts_won_lte_3_shots"].sum() / sb_s_tot["pts"].sum()

    # % BP Salvados en esa superficie
    pct_bp_s = ov_s["bp_saved"].sum() / ov_s["bk_pts"].sum()

    # % Pts Ganados Al Resto en esa superficie
    pct_resto_s = ov_s["return_pts_won"].sum() / ov_s["return_pts"].sum()

    # IV 1: Rally Corto Saque
    # Caída relativa: (% global saque - % rally corto) / % global saque
    caida_corto = (pct_saque_s - pct_corto_s) / pct_saque_s
    IV_corto = round(1.0 * caida_corto * ajuste, 3)

    # IV 2: Segundo Saque
    # Caída relativa: (% 1S - % 2S) / % 1S
    caida_2s = (pct_1s_s - pct_2s_s) / pct_1s_s
    IV_2s = round(1.0 * caida_2s * ajuste, 3)

    # IV 3: Rally Medio (row=4 y row=6 en esa superficie)
    # Requiere merge con matches para filtrar por superficie
    ro_s = dj_ro[dj_ro["match_id"].isin(match_ids_surf)]
    ro_medio_s = ro_s[ro_s["row"].isin(["4", "6"])]
    ro_total_s = ro_s[ro_s["row"] == "Total"]
    pct_medio_s = ro_medio_s["pts_won"].sum() / ro_medio_s["pts"].sum() if len(ro_medio_s) > 0 else np.nan
    pct_resto_ro_s = ro_total_s["pts_won"].sum() / ro_total_s["pts"].sum() if len(ro_total_s) > 0 else np.nan
    caida_medio = (pct_resto_ro_s - pct_medio_s) / pct_resto_ro_s if not np.isnan(pct_medio_s) else 0
    IV_medio = round(max(1.0 * caida_medio * ajuste, 0), 3)

    # IV 4: BP Bajo Presión
    # Caída relativa: (techo victorias 68.24% - actual) / techo victorias
    TECHO_BP_W = 0.6824  # referencia victorias Python validado
    caida_bp = (TECHO_BP_W - pct_bp_s) / TECHO_BP_W
    IV_bp = round(max(factor_conf * caida_bp * ajuste, 0), 3)

    resultados_iv[surf] = {
        "n_partidos": n_partidos, "factor_conf": factor_conf, "ajuste": ajuste,
        "IV_corto": IV_corto, "IV_2s": IV_2s, "IV_medio": IV_medio, "IV_bp": IV_bp,
        "pct_saque": pct_saque_s, "pct_1s": pct_1s_s, "pct_2s": pct_2s_s,
        "pct_corto": pct_corto_s, "pct_bp": pct_bp_s, "pct_medio": pct_medio_s
    }

print("=== ÍNDICE DE VULNERABILIDAD (IV) POR SUPERFICIE ===")
print(f"{'Vulnerabilidad':<30s} {'Hard':>8s} {'Clay':>8s} {'Grass':>8s}")
print("-" * 56)
for key, label in [("IV_corto", "IV Rally Corto Saque"),
                   ("IV_2s",   "IV Segundo Saque"),
                   ("IV_medio","IV Rally Medio Resto"),
                   ("IV_bp",   "IV BP Bajo Presión")]:
    vals = [resultados_iv[s][key] for s in ["Hard", "Clay", "Grass"]]
    print(f"  {label:<28s} {vals[0]:>8.3f} {vals[1]:>8.3f} {vals[2]:>8.3f}")

print()
print("Datos de soporte (Hard):")
h = resultados_iv["Hard"]
print(f"  Partidos Hard            : {h['n_partidos']}")
print(f"  % Pts Al Saque Hard      : {h['pct_saque']:.4f}")
print(f"  % Pts Rally Corto Hard   : {h['pct_corto']:.4f}")
print(f"  % Pts 1S Hard            : {h['pct_1s']:.4f}")
print(f"  % Pts 2S Hard            : {h['pct_2s']:.4f}")
print(f"  % Pts Rally Medio Hard   : {h['pct_medio']:.4f}")
print(f"  % BP Salvados Hard       : {h['pct_bp']:.4f}")

=== ÍNDICE DE VULNERABILIDAD (IV) POR SUPERFICIE ===
Vulnerabilidad                     Hard     Clay    Grass
--------------------------------------------------------
  IV Rally Corto Saque            0.482    0.506    0.258
  IV Segundo Saque                0.245    0.209    0.182
  IV Rally Medio Resto            0.133    0.090    0.090
  IV BP Bajo Presión              0.054    0.095    0.000

Datos de soporte (Hard):
  Partidos Hard            : 4849
  % Pts Al Saque Hard      : 0.6754
  % Pts Rally Corto Hard   : 0.3497
  % Pts 1S Hard            : 0.7394
  % Pts 2S Hard            : 0.5579
  % Pts Rally Medio Hard   : 0.3540
  % BP Salvados Hard       : 0.6453


## SECCIÓN 5 — Prioridad de Acción (PA)
Fórmula: `PA = IV × fortaleza_Fritz × factor_confianza_muestra`

Calculada para Hard court (superficie de referencia Fritz).

In [6]:
# Perfil de Fritz — fortalezas por dimensión táctica
# Reproducen exactamente la tabla Fritz_Perfil del DAX
fritz_perfil = {
    "Saque agresivo":         0.90,
    "Derecha ofensiva":       0.85,
    "Resto agresivo 2S":      0.80,
    "Punto corto (1-4g)":     0.85,
    "Reves defensivo":        0.55,
    "Juego en red":           0.50,
    "Rally largo (7+ golpes)": 0.40,
}

# Hard court — factor confianza = 1.0 (364 partidos)
h = resultados_iv["Hard"]
conf_hard = h["factor_conf"]

PA_corto  = round(h["IV_corto"] * fritz_perfil["Punto corto (1-4g)"]  * conf_hard, 3)
PA_2s     = round(h["IV_2s"]    * fritz_perfil["Resto agresivo 2S"]    * conf_hard, 3)
PA_medio  = round(h["IV_medio"] * fritz_perfil["Derecha ofensiva"]     * conf_hard, 3)
PA_bp     = round(h["IV_bp"]    * fritz_perfil["Saque agresivo"]       * conf_hard, 3)

# Semáforo de prioridad
def semaforo_pa(pa):
    if pa >= 0.35:  return "🟢 VERDE  — obligatoria"
    if pa >= 0.15:  return "🟡 AMARILLO — situacional"
    return              "🔴 ROJO   — no priorizar"

acciones = [
    ("PA Rally Corto (saque agresivo T/Wide)",  PA_corto),
    ("PA Segundo Saque (resto adelantado)",      PA_2s),
    ("PA Rally Medio (derecha en 4-6 golpes)",  PA_medio),
    ("PA BP (saque agresivo en break points)",  PA_bp),
]

print("=== PRIORIDAD DE ACCIÓN (PA) — Hard court ===")
print(f"{'Acción':<45s} {'PA':>6s}  Semáforo")
print("-" * 80)
for nombre, pa in sorted(acciones, key=lambda x: -x[1]):
    print(f"  {nombre:<43s} {pa:>6.3f}  {semaforo_pa(pa)}")

print()
print("RESUMEN DE INSTRUCCIONES TÁCTICAS:")
print("  1ª instrucción (PA más alta): servir agresivo T/Wide para forzar punto ≤3 golpes")
print("  2ª instrucción: adelantar posición en segundo saque de Djokovic")
print("  3ª instrucción: buscar derecha ofensiva en el 4º-6º golpe del rally")
print("  BP: usar saque agresivo en break points, no cambiar táctica bajo presión")

=== PRIORIDAD DE ACCIÓN (PA) — Hard court ===
Acción                                            PA  Semáforo
--------------------------------------------------------------------------------
  PA Rally Corto (saque agresivo T/Wide)       0.410  🟢 VERDE  — obligatoria
  PA Segundo Saque (resto adelantado)          0.196  🟡 AMARILLO — situacional
  PA Rally Medio (derecha en 4-6 golpes)       0.113  🔴 ROJO   — no priorizar
  PA BP (saque agresivo en break points)       0.049  🔴 ROJO   — no priorizar

RESUMEN DE INSTRUCCIONES TÁCTICAS:
  1ª instrucción (PA más alta): servir agresivo T/Wide para forzar punto ≤3 golpes
  2ª instrucción: adelantar posición en segundo saque de Djokovic
  3ª instrucción: buscar derecha ofensiva en el 4º-6º golpe del rally
  BP: usar saque agresivo en break points, no cambiar táctica bajo presión


## SECCIÓN 6 — Nivel de Riesgo (NR)
Fórmula: `NR = probabilidad_error_propio × coste_táctico`

Los parámetros de error de Fritz son estimaciones por perfil (fijos en la tabla Fritz_Perfil).

In [7]:
# Parámetros de error de Fritz (estimados por perfil)
prob_error = {
    "Saque agresivo":    0.15,  # 15% DF estimada bajo presión
    "Resto agresivo 2S": 0.20,  # 20% UFE por posición adelantada
    "Derecha rally 4-6": 0.18,  # 18% UFE en ese timing
    "Subida a red":      0.30,  # 30% error — Djokovic gana 70% en red
}

# Coste táctico
coste = {
    "normal":   1.0,
    "bp":       1.5,   # break point — coste máximo
    "timing":   1.2,   # timing crítico (4-6 golpes)
    "red":      1.3,   # subida mal ejecutada = punto gratis para Djokovic
}

NR_saque_normal = round(prob_error["Saque agresivo"]    * coste["normal"], 3)
NR_saque_bp     = round(prob_error["Saque agresivo"]    * coste["bp"],     3)
NR_resto_2s     = round(prob_error["Resto agresivo 2S"] * coste["normal"], 3)
NR_derecha      = round(prob_error["Derecha rally 4-6"] * coste["timing"], 3)
NR_red          = round(prob_error["Subida a red"]      * coste["red"],    3)

def semaforo_nr(nr):
    if nr < 0.15:  return "🟢 VERDE  — riesgo bajo"
    if nr < 0.25:  return "🟡 AMARILLO — riesgo medio"
    return             "🔴 ROJO   — riesgo alto"

niveles_riesgo = [
    ("NR Saque agresivo (normal)",   NR_saque_normal),
    ("NR Saque agresivo (BP)",       NR_saque_bp),
    ("NR Resto agresivo 2S",         NR_resto_2s),
    ("NR Derecha en rally 4-6 g",    NR_derecha),
    ("NR Subida a red",              NR_red),
]

print("=== NIVEL DE RIESGO (NR) ===")
print(f"{'Acción':<35s} {'NR':>6s}  Semáforo")
print("-" * 70)
for nombre, nr in sorted(niveles_riesgo, key=lambda x: -x[1]):
    print(f"  {nombre:<33s} {nr:>6.3f}  {semaforo_nr(nr)}")

print()
# Verificar contra % pts ganados en red por Djokovic
pct_red_dj = dj_net["pts_won"].sum() / dj_net["net_pts"].sum()
print(f"Nota: Djokovic gana {pct_red_dj:.1%} de los puntos en red.")
print(f"      El NR de subida a red ({NR_red:.3f}) refleja esa dificultad para Fritz.")

=== NIVEL DE RIESGO (NR) ===
Acción                                  NR  Semáforo
----------------------------------------------------------------------
  NR Subida a red                    0.390  🔴 ROJO   — riesgo alto
  NR Saque agresivo (BP)             0.225  🟡 AMARILLO — riesgo medio
  NR Derecha en rally 4-6 g          0.216  🟡 AMARILLO — riesgo medio
  NR Resto agresivo 2S               0.200  🟡 AMARILLO — riesgo medio
  NR Saque agresivo (normal)         0.150  🟡 AMARILLO — riesgo medio

Nota: Djokovic gana 70.2% de los puntos en red.
      El NR de subida a red (0.390) refleja esa dificultad para Fritz.


## SECCIÓN 7 — Tabla resumen consolidada
Tabla final para copiar en el documento ejecutivo del P3.

In [8]:
h = resultados_iv["Hard"]

resumen = [
    # (Patrón, IA, IV_hard, PA_hard, NR, Semáforo PA, Semáforo NR)
    ("Conversión BP (Djokovic)",     IA_bp,          h["IV_bp"],     PA_bp,    NR_saque_bp),
    ("Resto profundo (Djokovic)",    IA_resto_prof,  None,           None,     None),
    ("Rally largo 9+ golpes",        IA_rally_largo, None,           None,     None),
    ("Rally corto ≤3 golpes (Fritz)",None,           h["IV_corto"],  PA_corto, NR_saque_normal),
    ("Segundo saque (Djokovic)",     IA_saque_2s,    h["IV_2s"],     PA_2s,    NR_resto_2s),
    ("Rally medio 4-6 golpes",       None,           h["IV_medio"],  PA_medio, NR_derecha),
    ("Subida a red",                 None,           None,           None,     NR_red),
]

print("=== TABLA CONSOLIDADA DE ÍNDICES (Hard court) ===")
print(f"{'Patrón':<35s} {'IA':>6s} {'IV':>6s} {'PA':>6s} {'NR':>6s}")
print("-" * 65)
for patron, ia, iv, pa, nr in resumen:
    ia_s  = f"{ia:.3f}"  if ia  is not None else "  —  "
    iv_s  = f"{iv:.3f}"  if iv  is not None else "  —  "
    pa_s  = f"{pa:.3f}"  if pa  is not None else "  —  "
    nr_s  = f"{nr:.3f}"  if nr  is not None else "  —  "
    print(f"  {patron:<33s} {ia_s:>6s} {iv_s:>6s} {pa_s:>6s} {nr_s:>6s}")

print()
print("LEYENDA:")
print("  IA = Índice de Amenaza  [0-1]: qué daño hace Djokovic")
print("  IV = Índice de Vulnerabilidad [0-1]: dónde puede explotar Fritz")
print("  PA = Prioridad de Acción [0-1]: qué debe hacer Fritz primero")
print("  NR = Nivel de Riesgo [0-1]: coste de equivocarse")
print()
print("SEMÁFORO PA: 🟢 ≥0.35 | 🟡 0.15-0.35 | 🔴 <0.15")
print("SEMÁFORO NR: 🟢 <0.15 | 🟡 0.15-0.25 | 🔴 >0.25")

=== TABLA CONSOLIDADA DE ÍNDICES (Hard court) ===
Patrón                                  IA     IV     PA     NR
-----------------------------------------------------------------
  Conversión BP (Djokovic)           0.446  0.054  0.049  0.225
  Resto profundo (Djokovic)          0.541    —      —      —  
  Rally largo 9+ golpes              0.472    —      —      —  
  Rally corto ≤3 golpes (Fritz)        —    0.482  0.410  0.150
  Segundo saque (Djokovic)           0.359  0.245  0.196  0.200
  Rally medio 4-6 golpes               —    0.133  0.113  0.216
  Subida a red                         —      —      —    0.390

LEYENDA:
  IA = Índice de Amenaza  [0-1]: qué daño hace Djokovic
  IV = Índice de Vulnerabilidad [0-1]: dónde puede explotar Fritz
  PA = Prioridad de Acción [0-1]: qué debe hacer Fritz primero
  NR = Nivel de Riesgo [0-1]: coste de equivocarse

SEMÁFORO PA: 🟢 ≥0.35 | 🟡 0.15-0.35 | 🔴 <0.15
SEMÁFORO NR: 🟢 <0.15 | 🟡 0.15-0.25 | 🔴 >0.25


## SECCIÓN 8 — Validación por superficie (IV completo)
Tabla de índices IV para las tres superficies.

In [9]:
print("=== IV POR SUPERFICIE — tabla completa ===")
print(f"{'IV':<30s} {'Hard':>8s} {'Clay':>8s} {'Grass':>8s}  Nota")
print("-" * 75)

iv_keys = [
    ("IV_corto",  "IV Rally Corto Saque",  "Muestra robusta en todas"),
    ("IV_2s",     "IV Segundo Saque",      "Constante — no varía mucho"),
    ("IV_medio",  "IV Rally Medio Resto",  "Menor en Grass (bote bajo)"),
    ("IV_bp",     "IV BP Bajo Presión",    "Grass: muestra pequeña (58p)"),
]

for key, label, nota in iv_keys:
    vals = [resultados_iv[s][key] for s in ["Hard", "Clay", "Grass"]]
    print(f"  {label:<28s} {vals[0]:>8.3f} {vals[1]:>8.3f} {vals[2]:>8.3f}  {nota}")

print()
print("Partidos por superficie:")
for s in ["Hard", "Clay", "Grass"]:
    print(f"  {s:<8s}: {resultados_iv[s]['n_partidos']} partidos  "
          f"(confianza: {resultados_iv[s]['factor_conf']})")

=== IV POR SUPERFICIE — tabla completa ===
IV                                 Hard     Clay    Grass  Nota
---------------------------------------------------------------------------
  IV Rally Corto Saque            0.482    0.506    0.258  Muestra robusta en todas
  IV Segundo Saque                0.245    0.209    0.182  Constante — no varía mucho
  IV Rally Medio Resto            0.133    0.090    0.090  Menor en Grass (bote bajo)
  IV BP Bajo Presión              0.054    0.095    0.000  Grass: muestra pequeña (58p)

Partidos por superficie:
  Hard    : 4849 partidos  (confianza: 1.0)
  Clay    : 1877 partidos  (confianza: 1.0)
  Grass   : 837 partidos  (confianza: 1.0)
